In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.text("asset_id", "")
dbutils.widgets.text("assignment_id", "")
dbutils.widgets.text("rule_code", "")
dbutils.widgets.text("version_number", "")
dbutils.widgets.text("target_column_id", "")
dbutils.widgets.text("asset_execution_id", "")
dbutils.widgets.text("parameter_values", "")
dbutils.widgets.text("sql_expression", "")
dbutils.widgets.text("threshold_value", "")
dbutils.widgets.text("evaluation_operation", "")


asset_id = dbutils.widgets.get("asset_id").strip()
assignment_id = dbutils.widgets.get("assignment_id").strip()
rule_code = dbutils.widgets.get("rule_code").strip()
version_number = dbutils.widgets.get("version_number").strip()
target_column_id = dbutils.widgets.get("target_column_id").strip()
asset_execution_id = dbutils.widgets.get(
    "asset_execution_id"
).strip()
parameter_values = dbutils.widgets.get("parameter_values").strip()
sql_expression = dbutils.widgets.get("sql_expression").strip()
threshold_value = dbutils.widgets.get("threshold_value").strip()
evaluation_operation = dbutils.widgets.get("evaluation_operation").strip()


print("====================================")
print("RULE EXECUTION INPUT")
print("====================================")

print(f"asset_id             : {asset_id}")
print(f"assignment_id        : {assignment_id}")
print(f"rule_code            : {rule_code}")
print(f"version_number       : {version_number}")
print(f"target_column_id     : {target_column_id}")
print(f"parameter_values     : {parameter_values}")
print(f"sql_expression       : {sql_expression}")
print(f"threshold_value      : {threshold_value}")
print(f"evaluation_operation : {evaluation_operation}")

RULE EXECUTION INPUT
asset_id             : 28
assignment_id        : 62
rule_code            : C01
version_number       : 1
target_column_id     : 269
parameter_values     : 
sql_expression       : 
threshold_value      : 95
evaluation_operation : >=


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

import json
from decimal import Decimal
import pandas as pd
import re


CATALOG = "mastercard_card_241614"
SCHEMA = "default"

RULES_TABLE = f"{CATALOG}.{SCHEMA}.rules_m"
ASSIGNMENTS_TABLE = f"{CATALOG}.{SCHEMA}.rules_assignments_m"
ASSETS_TABLE = f"{CATALOG}.{SCHEMA}.assets_m"
ASSET_COLUMNS_TABLE = f"{CATALOG}.{SCHEMA}.asset_columns_m"

SOURCE_EXCEL = "/Volumes/mastercard_card_241614/default/test_data/PCN_DQ_TestDataset (1).xlsx"

SOURCE_VIEW = "dq_source_data"


print("====================================")
print("RULE EXECUTION CONFIGURATION")
print("====================================")

print(f"Rules table        : {RULES_TABLE}")
print(f"Assignments table  : {ASSIGNMENTS_TABLE}")
print(f"Assets table       : {ASSETS_TABLE}")
print(f"Columns table      : {ASSET_COLUMNS_TABLE}")
print(f"Source Excel       : {SOURCE_EXCEL}")
print(f"Source temp view   : {SOURCE_VIEW}")

RULE EXECUTION CONFIGURATION
Rules table        : mastercard_card_241614.default.rules_m
Assignments table  : mastercard_card_241614.default.rules_assignments_m
Assets table       : mastercard_card_241614.default.assets_m
Columns table      : mastercard_card_241614.default.asset_columns_m
Source Excel       : /Volumes/mastercard_card_241614/default/test_data/PCN_DQ_TestDataset (1).xlsx
Source temp view   : dq_source_data


In [0]:
print("====================================")
print("VALIDATING RULE EXECUTION INPUT")
print("====================================")

required_inputs = {
    "asset_id": asset_id,
    "assignment_id": assignment_id,
    "rule_code": rule_code,
    "version_number": version_number
}

missing_inputs = [
    name
    for name, value in required_inputs.items()
    if value == ""
]

if missing_inputs:
    raise ValueError(
        f"Missing required rule execution inputs: {missing_inputs}"
    )


try:
    asset_id_int = int(asset_id)
    assignment_id_int = int(assignment_id)
    version_number_int = int(version_number)
except ValueError:
    raise ValueError(
        "asset_id, assignment_id and version_number must be valid integers."
    )


if target_column_id != "":
    try:
        target_column_id_int = int(target_column_id)
    except ValueError:
        raise ValueError(
            "target_column_id must be a valid integer when provided."
        )
else:
    target_column_id_int = None


print("Input validation PASSED")
print(f"asset_id         : {asset_id_int}")
print(f"assignment_id    : {assignment_id_int}")
print(f"rule_code        : {rule_code}")
print(f"version_number   : {version_number_int}")
print(f"target_column_id : {target_column_id_int}")

VALIDATING RULE EXECUTION INPUT
Input validation PASSED
asset_id         : 28
assignment_id    : 62
rule_code        : C01
version_number   : 1
target_column_id : 269


In [0]:
print("====================================")
print("RESOLVING RULE ASSIGNMENT")
print("====================================")


rules_assignments_df = spark.table(ASSIGNMENTS_TABLE)


assignment_df = (
    rules_assignments_df
    .filter(F.col("assignment_id") == assignment_id_int)
    .filter(F.col("asset_id") == asset_id_int)
    .filter(F.col("rule_code") == rule_code)
    .filter(F.col("version_number") == version_number_int)
)


assignment_count = assignment_df.count()


if assignment_count == 0:
    raise ValueError(
        "No matching rule assignment found.\n"
        f"asset_id={asset_id_int}, "
        f"assignment_id={assignment_id_int}, "
        f"rule_code={rule_code}, "
        f"version_number={version_number_int}"
    )


if assignment_count > 1:
    raise ValueError(
        "Multiple matching rule assignments found. "
        "assignment_id should uniquely identify an assignment."
    )


assignment = assignment_df.first()


print("Rule assignment resolved successfully")
print("------------------------------------")

print(f"assignment_id        : {assignment['assignment_id']}")
print(f"asset_id             : {assignment['asset_id']}")
print(f"rule_code            : {assignment['rule_code']}")
print(f"version_number       : {assignment['version_number']}")
print(f"target_column_id     : {assignment['target_column_id']}")
print(f"target_type          : {assignment['target_type']}")
print(f"parameter_values     : {assignment['parameter_values']}")
print(f"sql_expression       : {assignment['sql_expression']}")
print(f"threshold_value      : {assignment['threshold_value']}")
print(f"evaluation_operation : {assignment['evaluation_operation']}")
print(f"is_enabled           : {assignment['is_enabled']}")
print(f"is_current           : {assignment['is_current']}")
# print("====================================")
# print("CHECKING CURRENT RULE ASSIGNMENTS")
# print("====================================")

# rules_assignments_df = spark.table(ASSIGNMENTS_TABLE)

# display(
#     rules_assignments_df
#     .filter(F.col("asset_id") == 28)
#     .filter(F.col("rule_code") == "C01")
#     .select(
#         "assignment_id",
#         "asset_id",
#         "rule_code",
#         "version_number",
#         "target_column_id",
#         "target_type",
#         "parameter_values",
#         "sql_expression",
#         "threshold_value",
#         "evaluation_operation",
#         "is_enabled",
#         "is_current",
#         "effective_from",
#         "effective_to"
#     )
#     .orderBy("assignment_id")
# )

RESOLVING RULE ASSIGNMENT
Rule assignment resolved successfully
------------------------------------
assignment_id        : 62
asset_id             : 28
rule_code            : C01
version_number       : 1
target_column_id     : 269
target_type          : COLUMN / TABLE
parameter_values     : None
sql_expression       : None
threshold_value      : 95.000000
evaluation_operation : >=
is_enabled           : True
is_current           : True


In [0]:
print("====================================")
print("RESOLVING RULE METADATA")
print("====================================")


rules_df = spark.table(RULES_TABLE)


rule_metadata_df = (
    rules_df
    .filter(F.col("rule_code") == rule_code)
    .filter(F.col("is_active") == True)
)


rule_metadata_count = rule_metadata_df.count()


if rule_metadata_count == 0:
    raise ValueError(
        f"No active rule metadata found for rule_code='{rule_code}'."
    )


if rule_metadata_count > 1:
    raise ValueError(
        f"Multiple active rule definitions found for "
        f"rule_code='{rule_code}'."
    )


rule_metadata = rule_metadata_df.first()


dimension = rule_metadata["dimension"]
rule_name = rule_metadata["rule_name"]
rule_type = rule_metadata["rule_type"]
description = rule_metadata["description"]
target_type = rule_metadata["target_type"]
parameter_definition = rule_metadata["parameter_definition"]
sql_query_template = rule_metadata["sql_query_template"]
default_threshold = rule_metadata["default_threshold"]
data_type = rule_metadata["data_type"]
semantic_type = rule_metadata["semantic_type"]


print("Rule metadata resolved successfully")
print("------------------------------------")

print(f"rule_code            : {rule_metadata['rule_code']}")
print(f"dimension            : {dimension}")
print(f"rule_name            : {rule_name}")
print(f"rule_type            : {rule_type}")
print(f"target_type          : {target_type}")
print(f"data_type            : {data_type}")
print(f"semantic_type        : {semantic_type}")
print(f"default_threshold    : {default_threshold}")
print(f"parameter_definition : {parameter_definition}")
print("------------------------------------")
print("SQL QUERY TEMPLATE")
print("------------------------------------")
print(sql_query_template)

RESOLVING RULE METADATA
Rule metadata resolved successfully
------------------------------------
rule_code            : C01
dimension            : COMPLETENESS
rule_name            : Completeness %
rule_type            : SQL
target_type          : COLUMN / TABLE
data_type            : ALL
semantic_type        : ANY
default_threshold    : 95.000000
parameter_definition : None
------------------------------------
SQL QUERY TEMPLATE
------------------------------------

SELECT
    100.0 * COUNT({column}) / NULLIF(COUNT(*), 0) AS metric_value,
    COUNT({column}) AS passed_count,
    COUNT(*) - COUNT({column}) AS failed_count,
    COUNT(*) AS total_count,
    COUNT(*) AS applicable_count
FROM {table}



In [0]:
print("====================================")
print("RESOLVING ASSET AND TARGET COLUMN")
print("====================================")


# --------------------------------------------------
# 1. Load asset metadata
# --------------------------------------------------

assets_df = spark.table(ASSETS_TABLE)

asset_df = (
    assets_df
    .filter(F.col("asset_id") == asset_id_int)
    .filter(F.col("is_active") == True)
)


asset_count = asset_df.count()


if asset_count == 0:
    raise ValueError(
        f"No active asset found for asset_id={asset_id_int}."
    )


if asset_count > 1:
    raise ValueError(
        f"Multiple active assets found for asset_id={asset_id_int}."
    )


asset = asset_df.first()

asset_name = asset["asset_name"]
asset_type = asset["asset_type"]
asset_s3_path = asset["s3_path"]
asset_domain_id = asset["domain_id"]
asset_sub_domain_id = asset["sub_domain_id"]


print("Asset resolved")
print("------------------------------------")
print(f"asset_id       : {asset_id_int}")
print(f"asset_name     : {asset_name}")
print(f"asset_type     : {asset_type}")
print(f"s3_path        : {asset_s3_path}")
print(f"domain_id      : {asset_domain_id}")
print(f"sub_domain_id  : {asset_sub_domain_id}")


# --------------------------------------------------
# 2. Resolve target column when provided
# --------------------------------------------------

target_column_name = None
target_column_data_type = None
target_column_nullable = None
target_column_metadata = None


if target_column_id_int is not None:

    asset_columns_df = spark.table(ASSET_COLUMNS_TABLE)

    target_column_df = (
        asset_columns_df
        .filter(F.col("column_id") == target_column_id_int)
        .filter(F.col("asset_id") == asset_id_int)
    )


    target_column_count = target_column_df.count()


    if target_column_count == 0:
        raise ValueError(
            "Target column could not be resolved.\n"
            f"column_id={target_column_id_int}, "
            f"asset_id={asset_id_int}"
        )


    if target_column_count > 1:
        raise ValueError(
            "Multiple target columns found for the same "
            "column_id and asset_id."
        )


    target_column = target_column_df.first()

    target_column_name = target_column["column_name"]
    target_column_data_type = target_column["data_type"]
    target_column_nullable = target_column["is_nullable"]
    target_column_metadata = target_column["metadata"]


    print("")
    print("Target column resolved")
    print("------------------------------------")
    print(f"column_id       : {target_column_id_int}")
    print(f"column_name     : {target_column_name}")
    print(f"data_type       : {target_column_data_type}")
    print(f"is_nullable     : {target_column_nullable}")
    print(f"metadata        : {target_column_metadata}")

else:

    print("")
    print("No target column specified.")
    print("Rule will execute at asset/table level.")

RESOLVING ASSET AND TARGET COLUMN
Asset resolved
------------------------------------
asset_id       : 28
asset_name     : 01_TXN_transaction_lifecycle
asset_type     : TABLE
s3_path        : source/TXN_PROCESSING/TRANSACTION_LIFECYCLE_STATUS/01_txn_transaction_lifecycle
domain_id      : 1
sub_domain_id  : 1

Target column resolved
------------------------------------
column_id       : 269
column_name     : DQ_FLAG
data_type       : STRING
is_nullable     : True
metadata        : {"semantic_type":"TEXT"}


In [0]:
# ============================================================
# VALIDATE TARGET COLUMN
# ============================================================

if target_column_id_int is not None:

    target_column_rows = (
        asset_columns_df
        .filter(
            (F.col("asset_id") == asset_id_int) &
            (F.col("column_id") == target_column_id_int)
        )
        .collect()
    )

    if not target_column_rows:

        raise ValueError(
            f"Target column ID {target_column_id_int} "
            f"does not exist for asset_id {asset_id_int}. "
            f"Assignment {assignment_id_int} "
            f"must be corrected in rules_assignments_m."
        )

    target_column_name = target_column_rows[0]["column_name"]

    print(
        f"Target column resolved : "
        f"{target_column_id_int} -> {target_column_name}"
    )

else:

    target_column_name = None

    print("Rule is table-level.")

Target column resolved : 269 -> DQ_FLAG


In [0]:
print("====================================")
print("LOADING SOURCE DATA")
print("====================================")


# --------------------------------------------------
# 1. Load the asset's Excel sheet
# --------------------------------------------------

print(f"Excel file : {SOURCE_EXCEL}")
print(f"Sheet name : {asset_name}")


pdf = pd.read_excel(
    SOURCE_EXCEL,
    sheet_name=asset_name
)


print(f"Rows loaded from Excel : {len(pdf)}")
print(f"Columns found           : {len(pdf.columns)}")


# --------------------------------------------------
# 2. Convert Pandas DataFrame to Spark DataFrame
# --------------------------------------------------

for col in pdf.columns:
    pdf[col] = pdf[col].apply(lambda x: str(x) if pd.notna(x) else None)

df = spark.createDataFrame(pdf)


print("")
print("Spark source DataFrame created")
print("------------------------------------")

print(f"Row count    : {df.count()}")
print(f"Column count : {len(df.columns)}")


# --------------------------------------------------
# 3. Register temporary SQL view
# --------------------------------------------------

df.createOrReplaceTempView(SOURCE_VIEW)


print("")
print(f"Temporary view created : {SOURCE_VIEW}")


# --------------------------------------------------
# 4. Display source structure
# --------------------------------------------------

print("")
print("SOURCE SCHEMA")
print("------------------------------------")

df.printSchema()

LOADING SOURCE DATA
Excel file : /Volumes/mastercard_card_241614/default/test_data/PCN_DQ_TestDataset (1).xlsx
Sheet name : 01_TXN_transaction_lifecycle
Rows loaded from Excel : 100
Columns found           : 14

Spark source DataFrame created
------------------------------------
Row count    : 100
Column count : 14

Temporary view created : dq_source_data

SOURCE SCHEMA
------------------------------------
root
 |-- TRANSACTION_ID: string (nullable = true)
 |-- LIFECYCLE_STAGE: string (nullable = true)
 |-- TRANSACTION_TYPE: string (nullable = true)
 |-- AMOUNT: string (nullable = true)
 |-- CURRENCY_CODE: string (nullable = true)
 |-- INITIATED_AT: string (nullable = true)
 |-- AUTHORIZED_AT: string (nullable = true)
 |-- CLEARED_AT: string (nullable = true)
 |-- SETTLED_AT: string (nullable = true)
 |-- CHANNEL: string (nullable = true)
 |-- IS_CROSS_BORDER: string (nullable = true)
 |-- NETWORK: string (nullable = true)
 |-- PAN_LAST4: string (nullable = true)
 |-- DQ_FLAG: string (

In [0]:
print("====================================")
print("PARSING RULE PARAMETERS")
print("====================================")


# --------------------------------------------------
# 1. Initialize parameter values
# --------------------------------------------------

parsed_parameters = {}


if parameter_values in ("", "null", "NULL", "None"):
    parsed_parameters = {}

else:
    try:
        parsed_parameters = json.loads(parameter_values)

    except json.JSONDecodeError as e:
        raise ValueError(
            "parameter_values is not valid JSON.\n"
            f"Received: {parameter_values}\n"
            f"Error: {str(e)}"
        )


# --------------------------------------------------
# 2. Validate parameter structure
# --------------------------------------------------

if not isinstance(parsed_parameters, dict):
    raise ValueError(
        "parameter_values must contain a JSON object."
    )


print("Parameter parsing completed")
print("------------------------------------")
print(f"Parameters supplied : {len(parsed_parameters)}")


if parsed_parameters:
    for parameter_name, parameter_value in parsed_parameters.items():
        print(
            f"{parameter_name} : "
            f"{parameter_value}"
        )
else:
    print("No parameters supplied.")


# --------------------------------------------------
# 3. Read parameter definition from rules_m
# --------------------------------------------------

parameter_definition_obj = None


if parameter_definition not in (None, "", "null", "NULL"):

    try:
        parameter_definition_obj = json.loads(
            parameter_definition
        )

    except json.JSONDecodeError as e:
        raise ValueError(
            f"parameter_definition for rule_code='{rule_code}' "
            "is not valid JSON.\n"
            f"Error: {str(e)}"
        )


# --------------------------------------------------
# 4. Validate required parameters
# --------------------------------------------------

if parameter_definition_obj:

    if isinstance(parameter_definition_obj, dict):

        parameter_definitions = (
            parameter_definition_obj.get(
                "parameters",
                []
            )
        )

    elif isinstance(parameter_definition_obj, list):

        parameter_definitions = parameter_definition_obj

    else:

        parameter_definitions = []


    for parameter in parameter_definitions:

        if not isinstance(parameter, dict):
            continue

        parameter_name = parameter.get("name")
        required = parameter.get("required", False)

        if (
            required
            and parameter_name
            and parameter_name not in parsed_parameters
        ):
            raise ValueError(
                f"Required parameter '{parameter_name}' "
                f"is missing for rule_code='{rule_code}'."
            )


print("")
print("Parameter validation PASSED")

PARSING RULE PARAMETERS
Parameter parsing completed
------------------------------------
Parameters supplied : 0
No parameters supplied.

Parameter validation PASSED


In [0]:
# ============================================================
# VALIDATE REQUIRED PARAMETERS
# ============================================================

print("====================================")
print("VALIDATING RULE PARAMETERS")
print("====================================")

required_parameters = []

if parameter_definition:
    try:
        parameter_definition_json = json.loads(
            parameter_definition
        )

        required_parameters = [
            p["name"]
            for p in parameter_definition_json.get("parameters", [])
            if p.get("required") is True
        ]

    except Exception as e:
        raise ValueError(
            f"Invalid parameter_definition JSON for rule "
            f"{rule_code}: {e}"
        )


missing_parameters = []

for parameter_name in required_parameters:

    value = parsed_parameters.get(parameter_name)

    if value is None:
        missing_parameters.append(parameter_name)

    elif isinstance(value, str) and not value.strip():
        missing_parameters.append(parameter_name)


if missing_parameters:

    raise ValueError(
        f"Rule {rule_code} is missing required parameters: "
        f"{', '.join(missing_parameters)}"
    )


print("Required parameters : ", required_parameters)
print("Provided parameters : ", list(parsed_parameters.keys()))
print("Parameter validation PASSED")

VALIDATING RULE PARAMETERS
Required parameters :  []
Provided parameters :  []
Parameter validation PASSED


In [0]:
print("====================================")
print("BUILDING EXECUTION CONTEXT")
print("====================================")


# ============================================================
# 1. Source relation
# ============================================================

source_relation = SOURCE_VIEW


# ============================================================
# 2. Safely quote target column
# ============================================================

target_column_sql = None

if target_column_name is not None:

    target_column_sql = (
        "`"
        + target_column_name.replace("`", "``")
        + "`"
    )


# ============================================================
# 3. Build base execution context
# ============================================================

execution_context = {
    "table": source_relation,
    "asset": source_relation,
}

if target_column_sql is not None:
    execution_context["column"] = target_column_sql


# ============================================================
# 4. Add parsed parameters
# ============================================================

for parameter_name, parameter_value in parsed_parameters.items():

    execution_context[parameter_name] = parameter_value


# ============================================================
# 5. Derived placeholder:
#    required_columns
#    ->
#    required_columns_condition
# ============================================================

if "required_columns" in parsed_parameters:

    required_columns = parsed_parameters["required_columns"]

    if not isinstance(required_columns, list):
        raise ValueError(
            "required_columns must be a JSON array of column names."
        )

    if len(required_columns) == 0:
        raise ValueError(
            "required_columns cannot be empty."
        )

    required_conditions = []

    for col in required_columns:

        required_conditions.append(
            f"`{col}` IS NOT NULL "
            f"AND TRIM(CAST(`{col}` AS STRING)) <> ''"
        )

    execution_context["required_columns_condition"] = (
        " AND ".join(required_conditions)
    )


# ============================================================
# 6. Derived placeholder:
#    selected_columns
#    ->
#    selected_columns_condition
# ============================================================

if "selected_columns" in parsed_parameters:

    selected_columns = parsed_parameters["selected_columns"]

    if not isinstance(selected_columns, list):
        raise ValueError(
            "selected_columns must be a JSON array of column names."
        )

    if len(selected_columns) == 0:
        raise ValueError(
            "selected_columns cannot be empty."
        )

    selected_conditions = []

    for col in selected_columns:

        selected_conditions.append(
            f"`{col}` IS NOT NULL "
            f"AND TRIM(CAST(`{col}` AS STRING)) <> ''"
        )

    execution_context["selected_columns_condition"] = (
        " AND ".join(selected_conditions)
    )


# ============================================================
# 7. Derived placeholder:
#    date_sequence_columns
#    ->
#    date_sequence_condition
# ============================================================

if "date_sequence_columns" in parsed_parameters:

    date_sequence_columns = parsed_parameters[
        "date_sequence_columns"
    ]

    if not isinstance(date_sequence_columns, list):
        raise ValueError(
            "date_sequence_columns must be a JSON array of column names."
        )

    if len(date_sequence_columns) < 2:
        raise ValueError(
            "date_sequence_columns must contain at least two columns."
        )

    date_conditions = []

    for i in range(len(date_sequence_columns) - 1):

        current_column = date_sequence_columns[i]
        next_column = date_sequence_columns[i + 1]

        date_conditions.append(
            f"`{current_column}` <= `{next_column}`"
        )

    execution_context["date_sequence_condition"] = (
        " AND ".join(date_conditions)
    )


# ============================================================
# 8. Display execution context
# ============================================================

print("Execution context created")
print("------------------------------------")

for key, value in execution_context.items():

    if key == "parameter_values":

        print(
            f"{key:30} : "
            f"{json.dumps(value, default=str)}"
        )

    else:

        print(
            f"{key:30} : {value}"
        )

BUILDING EXECUTION CONTEXT
Execution context created
------------------------------------
table                          : dq_source_data
asset                          : dq_source_data
column                         : `DQ_FLAG`


In [0]:
# ============================================================
# RENDER SQL TEMPLATE
# ============================================================

# print("====================================")
# print("RENDERING SQL TEMPLATE")
# print("====================================")

# if not sql_query_template:
#     raise ValueError(
#         f"No sql_query_template found for rule {rule_code}."
#     )


# rendered_sql = sql_query_template


# ------------------------------------------------------------
# Replace placeholders from execution context
# ------------------------------------------------------------

for placeholder, value in execution_context.items():

    token = "{" + placeholder + "}"

    if token in rendered_sql:

        if value is None:
            raise ValueError(
                f"Placeholder {token} requires a value."
            )

        rendered_sql = rendered_sql.replace(
            token,
            str(value)
        )


# ------------------------------------------------------------
# Check for unresolved placeholders
# ------------------------------------------------------------

import re

unresolved_placeholders = re.findall(
    r"\{([A-Za-z0-9_]+)\}",
    rendered_sql
)


if unresolved_placeholders:

    raise ValueError(
        f"Unresolved SQL placeholders for rule "
        f"{rule_code}: "
        f"{unresolved_placeholders}"
    )


print("")
print("RENDERED SQL")
print("------------------------------------")
print(rendered_sql)


# --------------------------------------------------
# Helper functions
# --------------------------------------------------

def sql_string(value):
    """
    Convert a Python value into a SQL string literal.
    """
    if value is None:
        return "NULL"

    value = str(value)

    return "'" + value.replace("'", "''") + "'"


def sql_identifier(value):
    """
    Safely quote a SQL identifier.
    """
    if value is None or str(value).strip() == "":
        raise ValueError("SQL identifier cannot be empty.")

    value = str(value).strip()

    return "`" + value.replace("`", "``") + "`"


def sql_number(value):
    """
    Validate and render a numeric SQL literal.
    """
    if value is None:
        raise ValueError("Numeric parameter cannot be NULL.")

    value_string = str(value).strip()

    if not re.fullmatch(
        r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)",
        value_string
    ):
        raise ValueError(
            f"Invalid numeric SQL value: {value_string}"
        )

    return value_string


def sql_multi_value(value):
    """
    Convert a list of values into a SQL IN-list.
    Example:
        ['USD', 'EUR']
        ->
        'USD','EUR'
    """

    if not isinstance(value, list):
        raise ValueError(
            "multi_value parameter must be a JSON array."
        )

    return ",".join(
        sql_string(item)
        for item in value
    )


# --------------------------------------------------
# Start with the rule template
# --------------------------------------------------

rendered_sql = sql_query_template


if rendered_sql is None or rendered_sql.strip() == "":
    raise ValueError(
        f"No SQL query template configured for "
        f"rule_code='{rule_code}'."
    )


# --------------------------------------------------
# Standard system placeholders
# --------------------------------------------------

standard_placeholders = {
    "{table}": source_relation,
    "{asset}": source_relation
}


if target_column_sql is not None:

    standard_placeholders["{column}"] = target_column_sql


for placeholder, replacement in standard_placeholders.items():

    rendered_sql = rendered_sql.replace(
        placeholder,
        replacement
    )


# --------------------------------------------------
# Parameter placeholders
# --------------------------------------------------

parameter_definitions = []

if parameter_definition_obj:

    if isinstance(parameter_definition_obj, dict):

        parameter_definitions = (
            parameter_definition_obj.get(
                "parameters",
                []
            )
        )

    elif isinstance(parameter_definition_obj, list):

        parameter_definitions = (
            parameter_definition_obj
        )


for parameter_definition_item in parameter_definitions:

    if not isinstance(parameter_definition_item, dict):
        continue

    parameter_name = parameter_definition_item.get("name")
    control = parameter_definition_item.get("control")

    if not parameter_name:
        continue

    if parameter_name not in parsed_parameters:
        continue

    value = parsed_parameters[parameter_name]


    # ----------------------------------------------
    # Column selector
    # ----------------------------------------------

    if control == "column_select":

        replacement = sql_identifier(value)


    # ----------------------------------------------
    # Multiple column selector
    # ----------------------------------------------

    elif control in (
        "column_multi_select",
        "ordered_column_select"
    ):

        if not isinstance(value, list):
            raise ValueError(
                f"Parameter '{parameter_name}' must be "
                "a JSON array of column names."
            )

        replacement = ",".join(
            sql_identifier(column)
            for column in value
        )


    # ----------------------------------------------
    # Numeric parameter
    # ----------------------------------------------

    elif control == "number":

        replacement = sql_number(value)


    # ----------------------------------------------
    # Text / regex
    # ----------------------------------------------

    elif control in (
        "text",
        "regex"
    ):

        replacement = sql_string(value)


    # ----------------------------------------------
    # Multiple literal values
    # ----------------------------------------------

    elif control == "multi_value":

        replacement = sql_multi_value(value)


    # ----------------------------------------------
    # Select / enum parameter
    # ----------------------------------------------

    elif control == "select":

        replacement = str(value).strip().upper()


    # ----------------------------------------------
    # Raw SQL expression
    # ----------------------------------------------

    elif control in (
        "expression",
        "expression_multi"
    ):

        if isinstance(value, list):

            replacement = " AND ".join(
                str(item)
                for item in value
            )

        else:

            replacement = str(value)


    # ----------------------------------------------
    # Datetime
    # ----------------------------------------------

    elif control == "datetime":

        replacement = (
            "TIMESTAMP "
            + sql_string(value)
        )


    # ----------------------------------------------
    # Asset selector
    # ----------------------------------------------

    elif control == "asset_select":

        raise ValueError(
            f"Cross-asset parameter '{parameter_name}' "
            "is not supported by the current Excel "
            "prototype source loader."
        )


    # ----------------------------------------------
    # JSON
    # ----------------------------------------------

    elif control == "json":

        replacement = sql_string(
            json.dumps(value)
            if not isinstance(value, str)
            else value
        )


    # ----------------------------------------------
    # Schema definition
    # ----------------------------------------------

    elif control == "schema_definition":

        raise ValueError(
            f"schema_definition parameter "
            f"'{parameter_name}' requires specialized "
            "schema rendering and is not supported "
            "in this first execution version."
        )


    else:

        raise ValueError(
            f"Unsupported parameter control "
            f"'{control}' for parameter "
            f"'{parameter_name}'."
        )


    placeholder = "{" + parameter_name + "}"


    rendered_sql = rendered_sql.replace(
        placeholder,
        replacement
    )


# --------------------------------------------------
# Custom SQL expression override
# --------------------------------------------------

if (
    rule_type == "CUSTOM_SQL"
    and effective_sql_expression
    and effective_sql_expression.strip() != ""
):

    rendered_sql = effective_sql_expression


# --------------------------------------------------
# Check for unresolved placeholders
# --------------------------------------------------

unresolved_placeholders = re.findall(
    r"\{[^{}]+\}",
    rendered_sql
)


if unresolved_placeholders:

    raise ValueError(
        "SQL template contains unresolved placeholders: "
        f"{sorted(set(unresolved_placeholders))}"
    )


# --------------------------------------------------
# Display final SQL
# --------------------------------------------------

print("SQL rendering completed")
print("------------------------------------")
print(rendered_sql)


RENDERED SQL
------------------------------------

SELECT
    100.0 * COUNT(`DQ_FLAG`) / NULLIF(COUNT(*), 0) AS metric_value,
    COUNT(`DQ_FLAG`) AS passed_count,
    COUNT(*) - COUNT(`DQ_FLAG`) AS failed_count,
    COUNT(*) AS total_count,
    COUNT(*) AS applicable_count
FROM dq_source_data

SQL rendering completed
------------------------------------

SELECT
    100.0 * COUNT(`DQ_FLAG`) / NULLIF(COUNT(*), 0) AS metric_value,
    COUNT(`DQ_FLAG`) AS passed_count,
    COUNT(*) - COUNT(`DQ_FLAG`) AS failed_count,
    COUNT(*) AS total_count,
    COUNT(*) AS applicable_count
FROM dq_source_data



In [0]:
print("====================================")
print("EXECUTING RULE SQL")
print("====================================")


# --------------------------------------------------
# 1. Execute rendered SQL
# --------------------------------------------------

try:

    rule_result_df = spark.sql(rendered_sql)

except Exception as e:

    raise RuntimeError(
        f"Rule SQL execution failed for "
        f"rule_code='{rule_code}'.\n"
        f"Error: {str(e)}"
    )


# --------------------------------------------------
# 2. Validate result structure
# --------------------------------------------------

result_columns = rule_result_df.columns

required_result_columns = [
    "metric_value",
    "passed_count",
    "failed_count",
    "total_count",
    "applicable_count"
]


missing_result_columns = [
    column
    for column in required_result_columns
    if column not in result_columns
]


if missing_result_columns:

    raise ValueError(
        "Rule SQL did not return the required "
        f"standard result columns: {missing_result_columns}\n"
        f"Returned columns: {result_columns}"
    )


# --------------------------------------------------
# 3. Validate result row count
# --------------------------------------------------

result_row_count = rule_result_df.count()


if result_row_count != 1:

    raise ValueError(
        "Rule SQL must return exactly one result row. "
        f"Returned {result_row_count} rows."
    )


rule_result = rule_result_df.first()


# --------------------------------------------------
# 4. Extract standardized metrics
# --------------------------------------------------

metric_value = (
    float(rule_result["metric_value"])
    if rule_result["metric_value"] is not None
    else None
)


passed_count = (
    int(rule_result["passed_count"])
    if rule_result["passed_count"] is not None
    else 0
)


failed_count = (
    int(rule_result["failed_count"])
    if rule_result["failed_count"] is not None
    else 0
)


total_count = (
    int(rule_result["total_count"])
    if rule_result["total_count"] is not None
    else 0
)


applicable_count = (
    int(rule_result["applicable_count"])
    if rule_result["applicable_count"] is not None
    else 0
)


# --------------------------------------------------
# 5. Calculate failure percentage
# --------------------------------------------------

if total_count > 0:

    failure_percentage = (
        100.0 * failed_count / total_count
    )

else:

    failure_percentage = 0.0


# --------------------------------------------------
# 6. Display result
# --------------------------------------------------

print("Rule SQL executed successfully")
print("------------------------------------")

print(f"metric_value       : {metric_value}")
print(f"passed_count       : {passed_count}")
print(f"failed_count       : {failed_count}")
print(f"total_count        : {total_count}")
print(f"applicable_count   : {applicable_count}")
print(f"failure_percentage : {failure_percentage:.2f}")

EXECUTING RULE SQL
Rule SQL executed successfully
------------------------------------
metric_value       : 100.0
passed_count       : 100
failed_count       : 0
total_count        : 100
applicable_count   : 100
failure_percentage : 0.00


In [0]:
print("====================================")
print("EVALUATING RULE RESULT")
print("====================================")


# --------------------------------------------------
# 1. Validate evaluation inputs
# --------------------------------------------------

if effective_threshold is None:

    evaluation_status = "NO_THRESHOLD"

    threshold_passed = None

    print("No threshold configured.")
    print("Rule executed successfully, but no PASS/FAIL evaluation was performed.")


elif metric_value is None:

    evaluation_status = "ERROR"

    threshold_passed = False

    print("Metric value is NULL.")
    print("Rule cannot be evaluated against the threshold.")


else:

    # ----------------------------------------------
    # 2. Apply evaluation operation
    # ----------------------------------------------

    if effective_evaluation_operation == ">=":

        threshold_passed = (
            metric_value >= effective_threshold
        )

    elif effective_evaluation_operation == "<=":

        threshold_passed = (
            metric_value <= effective_threshold
        )

    elif effective_evaluation_operation == ">":

        threshold_passed = (
            metric_value > effective_threshold
        )

    elif effective_evaluation_operation == "<":

        threshold_passed = (
            metric_value < effective_threshold
        )

    elif effective_evaluation_operation == "=":

        threshold_passed = (
            metric_value == effective_threshold
        )

    elif effective_evaluation_operation == "!=":

        threshold_passed = (
            metric_value != effective_threshold
        )

    else:

        raise ValueError(
            "Unsupported evaluation operation: "
            f"'{effective_evaluation_operation}'"
        )


    # ----------------------------------------------
    # 3. Determine status
    # ----------------------------------------------

    if threshold_passed:

        evaluation_status = "PASS"

    else:

        evaluation_status = "FAIL"


# --------------------------------------------------
# 4. Display evaluation
# --------------------------------------------------

print("")
print("RULE EVALUATION")
print("------------------------------------")

print(f"Metric value       : {metric_value}")
print(f"Threshold          : {effective_threshold}")
print(f"Evaluation         : {effective_evaluation_operation}")
print(f"Threshold passed   : {threshold_passed}")
print(f"Evaluation status  : {evaluation_status}")

EVALUATING RULE RESULT


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7304211177227297>, line 10
      3 print("====================================")
      6 # --------------------------------------------------
      7 # 1. Validate evaluation inputs
      8 # --------------------------------------------------
---> 10 if effective_threshold is None:
     12     evaluation_status = "NO_THRESHOLD"
     14     threshold_passed = None

NameError: name 'effective_threshold' is not defined

In [0]:
print("====================================")
print("WRITING RULE EXECUTION RESULT")
print("====================================")


# --------------------------------------------------
# 1. Build result details
# --------------------------------------------------

result_details = {
    "rule_code": rule_code,
    "rule_name": rule_name,
    "version_number": version_number_int,
    "target_column_id": target_column_id_int,
    "target_column_name": target_column_name,

    "metric_value": metric_value,
    "threshold_value": effective_threshold,
    "evaluation_operation": effective_evaluation_operation,

    "passed_count": passed_count,
    "failed_count": failed_count,
    "total_count": total_count,
    "applicable_count": applicable_count,

    "parameter_values": parsed_parameters
}


result_details_json = json.dumps(
    result_details,
    default=str
)


# --------------------------------------------------
# 2. Check for asset execution context
# --------------------------------------------------

asset_execution_id = globals().get(
    "asset_execution_id",
    None
)


if asset_execution_id in (
    None,
    "",
    "None"
):

    print(
        "No asset_execution_id supplied."
    )

    print(
        "Standalone test mode: "
        "result will not be inserted into "
        "dq_execution_results_m yet."
    )

    result_written = False


else:

    try:

        asset_execution_id_int = int(
            asset_execution_id
        )

    except ValueError:

        raise ValueError(
            "asset_execution_id must be a valid integer."
        )


    # --------------------------------------------------
    # 3. Prepare values
    # --------------------------------------------------

    failure_percentage_value = (
        Decimal(str(round(failure_percentage, 2)))
        if failure_percentage is not None
        else None
    )

    threshold_value_value = (
        Decimal(str(effective_threshold))
        if effective_threshold is not None
        else None
    )


    # --------------------------------------------------
    # 4. Insert using explicit target columns
    #
    # result_id is an IDENTITY column and is therefore
    # intentionally excluded.
    #
    # created_at is generated by the table default.
    # --------------------------------------------------

    result_df = spark.createDataFrame(
        [
            (
                asset_execution_id_int,
                rule_code,
                None,                         # rule_version_id - legacy field
                assignment_id_int,
                dimension,
                evaluation_status,
                int(total_count) if total_count is not None else None,
                int(passed_count) if passed_count is not None else None,
                int(failed_count) if failed_count is not None else None,
                failure_percentage_value,
                threshold_value_value,
                result_details_json
            )
        ],
        T.StructType([
            T.StructField(
                "asset_execution_id",
                T.LongType(),
                False
            ),
            T.StructField(
                "rule_code",
                T.StringType(),
                True
            ),
            T.StructField(
                "rule_version_id",
                T.LongType(),
                True
            ),
            T.StructField(
                "assignment_id",
                T.LongType(),
                True
            ),
            T.StructField(
                "dimension",
                T.StringType(),
                False
            ),
            T.StructField(
                "status",
                T.StringType(),
                False
            ),
            T.StructField(
                "records_evaluated",
                T.LongType(),
                True
            ),
            T.StructField(
                "records_passed",
                T.LongType(),
                True
            ),
            T.StructField(
                "records_failed",
                T.LongType(),
                True
            ),
            T.StructField(
                "failure_percentage",
                T.DecimalType(7, 2),
                True
            ),
            T.StructField(
                "threshold_value",
                T.DecimalType(18, 6),
                True
            ),
            T.StructField(
                "result_details",
                T.StringType(),
                True
            )
        ])
    )


    # --------------------------------------------------
    # 5. Write only the 12 result columns
    # --------------------------------------------------

    result_df.createOrReplaceTempView(
        "dq_rule_result_to_write"
    )


    spark.sql(f"""
        INSERT INTO {CATALOG}.{SCHEMA}.dq_execution_results_m
        (
            asset_execution_id,
            rule_code,
            rule_version_id,
            assignment_id,
            dimension,
            status,
            records_evaluated,
            records_passed,
            records_failed,
            failure_percentage,
            threshold_value,
            result_details
        )
        SELECT
            asset_execution_id,
            rule_code,
            rule_version_id,
            assignment_id,
            dimension,
            status,
            records_evaluated,
            records_passed,
            records_failed,
            failure_percentage,
            threshold_value,
            result_details
        FROM dq_rule_result_to_write
    """)


    result_written = True

    print(
        "Result successfully written to "
        "dq_execution_results_m."
    )


# --------------------------------------------------
# 6. Display result summary
# --------------------------------------------------

print("")
print("RESULT SUMMARY")
print("------------------------------------")

print(f"Rule code          : {rule_code}")
print(f"Assignment ID      : {assignment_id_int}")
print(f"Status             : {evaluation_status}")
print(f"Metric value       : {metric_value}")
print(f"Threshold          : {effective_threshold}")
print(f"Records evaluated  : {total_count}")
print(f"Records passed     : {passed_count}")
print(f"Records failed     : {failed_count}")
print(f"Result written     : {result_written}")

In [0]:
# ============================================================
# RETURN RESULT TO ORCHESTRATOR
# ============================================================

result = {
    "status": evaluation_status,
    "rule_code": rule_code,
    "assignment_id": assignment_id_int,
    "metric_value": metric_value,
    "threshold_value": effective_threshold,
    "evaluation_operation": effective_evaluation_operation,
    "records_evaluated": total_count,
    "records_passed": passed_count,
    "records_failed": failed_count,
    "failure_percentage": failure_percentage,
    "result_written": result_written
}

print("")
print("====================================")
print("RULE EXECUTION COMPLETE")
print("====================================")

print(json.dumps(result, default=str, indent=2))

dbutils.notebook.exit(
    json.dumps(result, default=str)
)